## Week 4 - Assignment 4 for COfDSAI Tutorial Markdown ##

This week we look to classify handwritten numbers to help a postal company know how accurate their software will be.

In [ ]:
"""
EBC4286 Assignment 4 -- i6194134 -- OvR squared-hinge kernel SVM on MNIST.

USAGE
    python main.py [mnist.npz] [--n_final 4000] [--n_grid 2000] [--n_vala 3000] [--n_valb 5000]
    All other settings are in the CONFIG block below. Nothing else needs editing.

WHAT IT DOES (in order)
    1. model selection on n_grid training images, scored on validation set A (never the test set):
       linear vs RBF, gamma x C grid, finer C-sweep at the best gamma, hinge/OvO reference (sklearn)
    2. learning curve: error versus n (n_final/8 ... n_final)
    3. final classifier on n_final images: primal AND dual solved for each of the 10 digit problems
       (ALL solving, including the model-selection fits, is done with CVXPY; see svmlib.py)
    4. evaluation: train error, error on fresh validation set B, test error (used once), confusion
    5. hard images from the dual variables

OUTPUTS (all rewritten on every run)
    results.tex     \\newcommand macros with every number used in the report (report.tex \\input's it)
    fig_main.pdf    C-sweep | learning curve | test confusion matrix
    fig_grid.pdf    validation error versus gamma for each C (model-selection figure)
    fig_hard.pdf    hardest training images and most confidently wrong test images
    keynumbers.csv  the key numbers of Appendix A

CACHING
    Every expensive fit is cached in ./cache/ under a key made of ALL settings that influence it
    (n, gamma, C, seeds, sizes, backend). Changing a setting therefore recomputes only what is new;
    nothing stale is ever reused. Delete ./cache to force a full recomputation.
"""
import sys, os, time, pickle, json, argparse, hashlib
import numpy as np
from load_mnist import load_mnist, subsample, confusion, key_numbers, show_digits
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time
import numpy as np
import scipy.linalg as sl
import cvxpy as cp

# ============================== CONFIG ==============================
#ap = argparse.ArgumentParser()
#ap.add_argument("--n_final", type=int, default=1000)    # training images of the final classifier
#ap.add_argument("--n_grid", type=int, default=500)     # training images used for model selection
#ap.add_argument("--n_vala", type=int, default=3000)     # validation set A: choose gamma, C
#ap.add_argument("--n_valb", type=int, default=5000)     # validation set B: reported val_error only
#ap.add_argument("--solver", default="CLARABEL")         # CVXPY solver: CLARABEL (default), SCS, OSQP, ...
#args = ap.parse_args()
n_final = 4000
n_grid = 500
n_vala = 5000
n_valb = 10000

STUDENT = "i6194134"
GAMMAS = [0.005, 0.01, 0.02, 0.04, 0.08, 0.16]          # gaussian/RBF gamma grid
CS_RBF = [1, 10, 100]                                   # C grid for the RBF
CS_LIN = [0.001, 0.01, 0.1, 1]                          # C grid for the linear kernel
CS_SWEEP = [0.1, 0.3, 1, 3, 10, 30, 100, 300]           # fine C-sweep at the best gamma
SEED_SPLIT, SEED_TRAIN = 1, 0                           # split of validation sets / draw of training images
NG, NF = n_grid, n_final
NS = [NF // 8, NF // 4, NF // 2, NF]                    # learning-curve sizes
# =====================================================================

os.makedirs("cache", exist_ok=True)
X, y, Xt, yt = load_mnist()

# disjoint sets, all from the TRAINING data: validation A, validation B, pool for training images
pool = np.random.default_rng(SEED_SPLIT).permutation(60000)
valA = pool[:n_vala]
valB = pool[n_vala:n_vala + n_valb]
rest = pool[n_vala + n_valb:]
assert not (set(valA) & set(valB)) and not (set(rest) & (set(valA) | set(valB)))

"""
svmlib.py -- squared-hinge (L2-loss) kernel SVM: primal and Lagrange dual, BOTH SOLVED WITH CVXPY.

Primal (K + eps*I = L L'):
    min_{u,b,xi}  1/2 |u|^2 + C sum xi_i^2     s.t.  y_i (L_i u + b) >= 1 - xi_i
Dual (derived by hand in the report), H = (y y') o K_eps + I/(2C):
    max_{alpha}   1'alpha - 1/2 alpha' H alpha   s.t.  alpha >= 0,  y'alpha = 0
      (written below with H alpha = L'(y o alpha) so that CVXPY sees a sum of squares)
Recovery:   c = alpha o y,   f(x) = sum_i c_i k(x_i, x) + b,   xi = alpha/(2C).
"""
          # required: this module does all solving through CVXPY

EPS = 1e-6                  # jitter: K_eps = K + EPS*I is positive definite; used in BOTH problems
SOLVER = "CLARABEL"         # CVXPY solver name; set from main.py (--solver)


# ---------------------------------------------------------------- kernels and factor
def sqdist(A, B):
    return np.maximum((A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0)


def kernel(A, B, kind="rbf", gamma=0.02):
    if kind == "linear":
        return A @ B.T
    return np.exp(-gamma * sqdist(A, B))


def factor(K):
    """K_eps = K + EPS I and its Cholesky factor L (K_eps = L L')."""
    Keps = K + EPS * np.eye(len(K))
    return Keps, np.linalg.cholesky(Keps)


# ---------------------------------------------------------------- the two problems
def solve_primal(L, y, C):
    n = len(y)
    u = cp.Variable(n); b = cp.Variable(); xi = cp.Variable(n)
    margin = cp.multiply(y, L @ u + b) >= 1 - xi
    prob = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(u) + C * cp.sum_squares(xi)), [margin])
    prob.solve(solver=SOLVER)
    prob.is_dcp(dpp=True)
    return u.value, float(b.value), float(prob.value), prob.status


def dual_problem(L, C):
    """Dual with y as a CVXPY Parameter, so one problem object serves all ten digits (compiled once)."""
    n = L.shape[0]
    a = cp.Variable(n)
    yp = cp.Parameter(n)
    obj = cp.sum(a) - 0.5 * cp.sum_squares(L.T @ cp.multiply(yp, a)) - cp.sum_squares(a) / (4 * C)
    return cp.Problem(cp.Maximize(obj), [a >= 0, yp @ a == 0]), a, yp


def solve_dual(prob, a, yp, y):
    yp.value = y
    prob.solve(solver=SOLVER)
    prob.is_dcp(dpp=True)
    return np.maximum(a.value, 0.0), float(prob.value), prob.status


# ---------------------------------------------------------------- recovery from the dual
def recover(Keps, y, alpha, C):
    """Kernel coefficients c = alpha o y and intercept b from complementary slackness (alpha_i > 0)."""
    c = alpha * y
    u_f = Keps @ c                                  # = L u, the scores without intercept
    sv = alpha > 1e-8 * max(alpha.max(), 1e-300)
    b = np.mean(y[sv] * (1 - alpha[sv] / (2 * C)) - u_f[sv]) if sv.any() else 0.0
    return c, b


# ---------------------------------------------------------------- model selection: OvR, dual only
def fit_ovr_dual(Keps, L, ylab, C):
    """10 one-versus-rest duals with CVXPY (one compiled problem, y swapped per digit)."""
    prob, a, yp = dual_problem(L, C)
    cs, bs = [], []
    for k in range(10):
        y = np.where(ylab == k, 1.0, -1.0)
        alpha, _, status = solve_dual(prob, a, yp, y)
        c, b = recover(Keps, y, alpha, C)
        cs.append(c); bs.append(b)
    return np.array(cs), np.array(bs)


# ---------------------------------------------------------------- final: one digit, primal AND dual
def solve_binary(Keps, L, y, C):
    """Solve primal and dual of one binary problem with CVXPY; return everything needed for the checks."""
    out = {}
    t = time.perf_counter(); u, b_p, pv, ps = solve_primal(L, y, C); out["tp"] = time.perf_counter() - t
    prob, a, yp = dual_problem(L, C)
    t = time.perf_counter(); alpha, dv, ds = solve_dual(prob, a, yp, y); out["td"] = time.perf_counter() - t
    c_d, b_d = recover(Keps, y, alpha, C)
    c_p = sl.solve_triangular(L.T, u, lower=False)  # c = L^{-T} u
    out.update(pv=pv, dv=dv, gap=abs(pv - dv), alpha=alpha, c_p=c_p, b_p=b_p, c_d=c_d, b_d=b_d,
               status=(ps, ds))
    return out

def draw(n):
    return rest[subsample(y[rest], n, SEED_TRAIN)]       # balanced; subsets are nested in n


def cache(name, key, fn):
    f = "cache/%s_%s.pkl" % (name, hashlib.md5(repr(key).encode()).hexdigest()[:12])
    if os.path.exists(f):
        return pickle.load(open(f, "rb"))
    r = fn(); pickle.dump(r, open(f, "wb")); return r


def err(p, t):
    return 100 * np.mean(p != t)


def predict(cs, bs, Xs, Xnew, kind, gamma):
    return np.argmax(kernel(Xnew, Xs, kind, gamma) @ cs.T + bs, axis=1)


def fit_ovr(idx, kind, gamma, C):
    """OvR fit for model selection: ten duals, each solved with CVXPY. Returns coefficients, intercepts."""
    Xs, ys = X[idx], y[idx]
    Keps, L = factor(kernel(Xs, Xs, kind, gamma))
    return fit_ovr_dual(Keps, L, ys, C)


def fit_eval(n, kind, gamma, C):
    """(train error %, validation-A error %, seconds) of one OvR fit; cached per setting."""
    key = (n, kind, gamma, C, SEED_SPLIT, SEED_TRAIN, n_vala, "CLARABEL")
    def run():
        idx = draw(n); t = time.time()
        cs, bs = fit_ovr(idx, kind, gamma, C); tt = time.time() - t
        return (err(predict(cs, bs, X[idx], X[idx], kind, gamma), y[idx]),
                err(predict(cs, bs, X[idx], X[valA], kind, gamma), y[valA]), tt)
    r = cache("fit", key, run); print("fit", key[:4], "-> train %.2f val %.2f (%.0fs)" % r, flush=True)
    return r


se = lambda p, n: 100 * np.sqrt(p / 100 * (1 - p / 100) / n)   # binomial standard error, in points


In [34]:

# =============== 1. model selection (n = NG) ===============
lin = {C: fit_eval(NG, "linear", 0, C) for C in CS_LIN}
grid = {(g, C): fit_eval(NG, "rbf", g, C) for g in GAMMAS for C in CS_RBF}
(GAMMA, CG), gbest = min(grid.items(), key=lambda kv: kv[1][1])
gam_curve = {g: min(grid[(g, C)][1] for C in CS_RBF) for g in GAMMAS}          # best-C validation error per gamma
se_g = se(grid[(GAMMA, CG)][1], n_vala)
gam_ok = [g for g in GAMMAS if gam_curve[g] <= grid[(GAMMA, CG)][1] + se_g]      # within one SE of the best
bestlin = min(lin.items(), key=lambda kv: kv[1][1])

csweep = [(C,) + fit_eval(NG, "rbf", GAMMA, C)[:2] for C in CS_SWEEP]
CBEST, _, vC = min(csweep, key=lambda r: r[2])
se_c = se(vC, n_vala)
c_ok = [r[0] for r in csweep if r[2] <= vC + se_c]
c_zero = min((r[0] for r in csweep if r[1] == 0), default=None)
print("best gamma", GAMMA, "best C", CBEST, "| gammas within 1 SE:", gam_ok, "| Cs within 1 SE:", c_ok)


def stage_ref():
    
    idx = draw(NG); r = {}
    for nm, m in (("ovo", SVC(kernel="rbf", gamma=GAMMA, C=CBEST * 2)),
                  ("ovr", OneVsRestClassifier(SVC(kernel="rbf", gamma=GAMMA, C=CBEST * 2)))):
        m.fit(X[idx], y[idx]); r[nm] = err(m.predict(X[valA]), y[valA])
    return r
ref = cache("ref", (NG, GAMMA, CBEST, SEED_SPLIT, SEED_TRAIN, n_vala), stage_ref)  # scikit-learn reference models (hinge/OvO), NOT our SVM

ncurve = [(n,) + fit_eval(n, "rbf", GAMMA, CBEST) for n in NS]                    # (n, train, val, seconds)


fit (500, 'linear', 0, 0.001) -> train 11.80 val 19.26 (5s)


fit (500, 'linear', 0, 0.01) -> train 3.00 val 16.72 (5s)
fit (500, 'linear', 0, 0.1) -> train 0.00 val 18.38 (5s)
fit (500, 'linear', 0, 1) -> train 0.00 val 20.58 (7s)
fit (500, 'rbf', 0.005, 1) -> train 1.60 val 14.54 (4s)
fit (500, 'rbf', 0.005, 10) -> train 0.00 val 13.42 (5s)
fit (500, 'rbf', 0.005, 100) -> train 0.00 val 13.80 (5s)
fit (500, 'rbf', 0.01, 1) -> train 0.40 val 13.08 (3s)
fit (500, 'rbf', 0.01, 10) -> train 0.00 val 12.36 (5s)
fit (500, 'rbf', 0.01, 100) -> train 0.00 val 12.44 (5s)
fit (500, 'rbf', 0.02, 1) -> train 0.00 val 11.88 (4s)
fit (500, 'rbf', 0.02, 10) -> train 0.00 val 11.40 (4s)
fit (500, 'rbf', 0.02, 100) -> train 0.00 val 11.38 (4s)
fit (500, 'rbf', 0.04, 1) -> train 0.00 val 12.06 (4s)
fit (500, 'rbf', 0.04, 10) -> train 0.00 val 11.30 (4s)
fit (500, 'rbf', 0.04, 100) -> train 0.00 val 11.28 (4s)
fit (500, 'rbf', 0.08, 1) -> train 0.00 val 15.92 (3s)
fit (500, 'rbf', 0.08, 10) -> train 0.00 val 15.32 (4s)
fit (500, 'rbf', 0.08, 100) -> train 0.00 va

In [ ]:

# =============== 2. final classifier: primal AND dual per digit ===============
def stage_final():
    idx = draw(NF); Xs, ys = X[idx], y[idx]
    t0 = time.perf_counter(); Keps, L = factor(kernel(Xs, Xs, "rbf", GAMMA)); tchol = time.perf_counter() - t0
    R = {"idx": idx, "tchol": tchol, "probs": []}
    for k in range(10):
        o = solve_binary(Keps, L, np.where(ys == k, 1.0, -1.0), CBEST)
        R["probs"].append(o)
        print("final digit", k, "tp %.1f td %.1f gap %.2e" % (o["tp"], o["td"], o["gap"]), flush=True)
    return R
fin = cache("final", (NF, GAMMA, CBEST, "CLARABEL", EPS, SEED_TRAIN, SEED_SPLIT, n_vala,  n_valb), stage_final)

idx = fin["idx"]; Xs, ys = X[idx], y[idx]; P = fin["probs"]
Cd = np.array([p["c_d"] for p in P]); bd = np.array([p["b_d"] for p in P])
Cp = np.array([p["c_p"] for p in P]); bp = np.array([p["b_p"] for p in P])
pred = lambda C_, b_, Z: predict(C_, b_, Xs, Z, "rbf", GAMMA)
tr_d = pred(Cd, bd, Xs)
vB_d = pred(Cd, bd, X[valB])
te_d, te_p = pred(Cd, bd, Xt), pred(Cp, bp, Xt)          # test set used ONCE, only for the final classifier
kn_tr, kn_val, kn_te = key_numbers(ys, tr_d), key_numbers(y[valB], vB_d), key_numbers(yt, te_d)
M = confusion(yt, te_d)
agree = 100 * np.mean(te_d == te_p)
gap_max = max(p["gap"] for p in P); gap_rel = max(p["gap"] / abs(p["pv"]) for p in P)
rec = max(np.linalg.norm(p["c_p"] - p["c_d"]) / np.linalg.norm(p["c_d"]) for p in P)
tp_tot = fin["tchol"] + sum(p["tp"] for p in P); td_tot = sum(p["td"] for p in P)
status = all(p["status"] == ("optimal", "optimal") for p in P)
print(kn_tr, kn_val, kn_te, "| agree", agree, "gap", gap_max, "rec", rec, "tp", tp_tot, "td", td_tot, "status", status)


final digit 0 tp 139.7 td 143.3 gap 2.39e-07
final digit 1 tp 156.8 td 145.2 gap 1.82e-07
final digit 2 tp 155.1 td 131.6 gap 8.88e-07
final digit 3 tp 148.5 td 139.1 gap 1.97e-06
final digit 4 tp 169.0 td 148.6 gap 2.78e-07
final digit 5 tp 160.0 td 137.3 gap 1.84e-06
final digit 6 tp 165.0 td 143.3 gap 1.09e-07
final digit 7 tp 158.7 td 127.3 gap 1.24e-07
final digit 8 tp 129.8 td 109.2 gap 2.30e-06
final digit 9 tp 137.8 td 128.6 gap 1.14e-06
{'error': 0.0, 'confused_pair': '0-0', 'confused_n': 0} {'error': 4.51, 'confused_pair': '3-5', 'confused_n': 35} {'error': 4.01, 'confused_pair': '2-7', 'confused_n': 31} | agree 100.0 gap 2.302336923776238e-06 rec 1.3710317814597015e-05 tp 1521.6373035999713 td 1353.4195928000845 status True


In [42]:

# =============== 3. hard images from the dual variables ===============
A = np.array([p["alpha"] for p in P])                    # 10 x n; alpha_i^(k) = 2C xi_i^(k)
hard = A.sum(0) / (2 * CBEST)                            # total squared-hinge slack of image i
order = np.argsort(-hard)
tol = 1e-8 * A.max()
frac_any = 100 * np.mean(A.max(0) > tol)                 # images with some alpha > 0
frac_zero = 100 - frac_any
share10 = 100 * hard[order[:NF // 10]].sum() / hard.sum()
top = order[:16]
top_mis = int(np.sum(tr_d[top] != ys[top]))
Ft = kernel(Xt, Xs, "rbf", GAMMA) @ Cd.T + bd
wrong = np.where(te_d != yt)[0]
wtop = wrong[np.argsort(-(Ft[wrong, te_d[wrong]] - Ft[wrong, yt[wrong]]))[:16]]



In [40]:

# =============== figures ===============

plt.rcParams.update({"font.size": 8})

# --- fig_main
fig, ax = plt.subplots(1, 3, figsize=(7.4, 1.95), gridspec_kw={"width_ratios": [1, 1, 1.05]})
c = np.array(csweep)
ax[0].semilogx(c[:, 0], c[:, 1], "o-", label="train"); ax[0].semilogx(c[:, 0], c[:, 2], "s-", label="validation")
ax[0].axvline(CBEST, color="k", ls="--", lw=0.8); ax[0].plot([CBEST], [vC], "k*", ms=9, zorder=5)
ax[0].annotate("chosen $C$=%g\n(val. %.1f%%)" % (CBEST, vC), (CBEST, vC), xytext=(c[0, 0] * 1.2, vC * 0.62),
               fontsize=7, arrowprops=dict(arrowstyle="->", lw=0.6))
ax[0].set_xlabel("$C$ (n=%d, $\\gamma$=%g)" % (NG, GAMMA)); ax[0].set_ylabel("error (%)")
ax[0].legend(frameon=False, fontsize=7, loc="upper center"); ax[0].grid(alpha=.3)
n_ = np.array(ncurve)
ax[1].semilogx(n_[:, 0], n_[:, 1], "o-", label="train"); ax[1].semilogx(n_[:, 0], n_[:, 2], "s-", label="validation")
ax[1].set_xticks(NS); ax[1].set_xticklabels([str(v) for v in NS]); ax[1].minorticks_off()
ax[1].set_xlabel("training images $n$ ($C$=%g)" % CBEST); ax[1].grid(alpha=.3); ax[1].legend(frameon=False)
# Replace the ax[2].imshow(...) and text loop in fig_main with:
M_off = M.copy().astype(float)
np.fill_diagonal(M_off, np.nan)
vmax_off = np.nanmax(M_off)

cmap = plt.cm.Blues.copy()
cmap.set_bad(color="#1f3b73")  # dark navy for diagonal cells
ax[2].imshow(M_off, cmap=cmap, vmin=0, vmax=vmax_off)
for i in range(10):
    for j in range(10):
        if i != j and M[i, j] > 0:
            ax[2].text(j, i, M[i, j], ha="center", va="center", fontsize=5.5,
                       color="w" if M[i, j] > 0.55 * vmax_off else "k")
                       
ax[2].set_xticks(range(10)); ax[2].set_yticks(range(10)); ax[2].set_xlabel("predicted"); ax[2].set_ylabel("true")
ax[2].set_title("test confusion (off-diagonal)", fontsize=7)
fig.tight_layout(pad=0.3); fig.savefig("fig_main.pdf"); plt.close(fig)

# --- fig_grid: model selection
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.5))
for C in CS_RBF:
    ax[0].semilogx(GAMMAS, [grid[(g, C)][1] for g in GAMMAS], "o-", label="RBF, $C$=%g" % C)
    # In fig_grid, swap the two ax[1].semilogx calls so blue=train, orange=validation:
    ax[1].semilogx(c[:, 0], c[:, 1], "o-", label="train")
    ax[1].semilogx(c[:, 0], c[:, 2], "s-", label="validation")
ax[0].axhline(bestlin[1][1], color="gray", ls="--", label="best linear ($C$=%g)" % bestlin[0])
ax[0].axhspan(grid[(GAMMA, CG)][1] - se_g, grid[(GAMMA, CG)][1] + se_g, color="k", alpha=.08, label="best $\\pm$1 SE")
ax[0].axvline(GAMMA, color="k", ls=":", lw=.8)
ax[0].set_xlabel("$\\gamma$ (n=%d)" % NG); ax[0].set_ylabel("validation-A error (%)"); ax[0].grid(alpha=.3)
ax[0].legend(frameon=False, fontsize=6.5)
ax[1].set_title("train and validation error at best $\\gamma$", fontsize=7)
ax[1].semilogx(c[:, 0], c[:, 2], "s-", label="validation"); ax[1].semilogx(c[:, 0], c[:, 1], "o-", label="train")
ax[1].axhspan(vC - se_c, vC + se_c, color="k", alpha=.08, label="best $\\pm$1 SE")
ax[1].axvline(CBEST, color="k", ls=":", lw=.8)
ax[1].set_xlabel("$C$ ($\\gamma$=%g)" % GAMMA); ax[1].grid(alpha=.3); ax[1].legend(frameon=False, fontsize=6.5)
fig.tight_layout(pad=0.4); fig.savefig("fig_grid.pdf"); plt.close(fig)

# --- fig_hard
imgs = np.vstack([Xs[top], Xt[wtop]])
tit = [f"{ys[i]}→{tr_d[i]}" for i in top] + [f"{yt[i]}→{te_d[i]}" for i in wtop]
show_digits(imgs, titles=tit, ncols=16, size=0.62).savefig("fig_hard.pdf"); plt.close("all")

# =============== macros for the report ===============
rf = lambda v, d=2: f"{v:.{d}f}"
def sci(v):
    m, e = f"{v:.1e}".split("e"); return f"{m}\\times10^{{{int(e)}}}"
lst = lambda v: ",".join(f"{x:g}" for x in v)
off = sorted([(M[i, j] + M[j, i], i, j) for i in range(10) for j in range(i + 1, 10)], reverse=True)[:3]
pd_err = 100 * (M.sum(1) - np.diag(M)) / M.sum(1)
pd_order = np.argsort(-pd_err)
te_err, val_err = kn_te["error"], kn_val["error"]
dbl = (n_[-1, 3] / n_[0, 3]) ** (1 / np.log2(n_[-1, 0] / n_[0, 0]))               # time factor per doubling of n
lastgain = n_[-2, 2] - n_[-1, 2]
mac = {
 "student": STUDENT, "NF": NF, "perdig": NF // 10, "NG": NG, "nvala": n_vala, "nvalb": n_valb,
 "gamlist": lst(GAMMAS), "Clist": lst(CS_RBF), "linClist": lst(CS_LIN),
 "Gb": f"{GAMMA:g}", "Cb": f"{CBEST:g}", "gamlo": f"{min(gam_ok):g}", "gamhi": f"{max(gam_ok):g}",
 "gamedge": ("the best $\\gamma$ is at the edge of the grid, so a larger one might help slightly"
             if GAMMA in (min(GAMMAS), max(GAMMAS)) else "the best $\\gamma$ lies inside the grid"),
 "Clo": f"{min(c_ok):g}", "Chi": f"{max(c_ok):g}", "Czero": f"{c_zero:g}" if c_zero is not None else "none",
 "linerr": rf(bestlin[1][1]), "linC": f"{bestlin[0]:g}", "rbferr": rf(grid[(GAMMA, CG)][1]),
 "ovo": rf(ref["ovo"]), "ovr": rf(ref["ovr"]), "valA": rf(vC), "seA": rf(se_c),
 "nlo": f"{NS[0]}", "nhi": f"{NS[-1]}", "vlo": rf(n_[0, 2], 1), "vmid": rf(n_[-2, 2], 1), "vhi": rf(n_[-1, 2], 1),
 "nmid": f"{NS[-2]}", "tlo": rf(n_[0, 3], 0), "tmid": rf(n_[-2, 3], 0), "thi": rf(n_[-1, 3], 0),
 "dbl": rf(dbl, 0), "lastgain": rf(lastgain, 1), "dbltime": rf(td_tot * dbl / 60, 0),
 "trerr": rf(kn_tr["error"]), "valerr": rf(val_err), "valerrone": rf(val_err, 1),
 "teerr": rf(te_err), "teerrone": rf(te_err, 1),
 "sevalb": rf(se(val_err, n_valb)), "sete": rf(se(te_err, len(yt))), "ci": rf(1.96 * se(te_err, len(yt)), 1),
 "pair": kn_te["confused_pair"], "pairn": kn_te["confused_n"],
 "topconf": ", ".join(f"{i}--{j} ({n})" for n, i, j in off), "topslash": ", ".join(f"{i}/{j}" for n, i, j in off[:3]),
 "worstdig": ", ".join(f"{d} ({pd_err[d]:.1f}\\%)" for d in pd_order[:3]), "worstlist": ", ".join(str(d) for d in pd_order[:3]),
 "bestdig": ", ".join(f"{d} ({pd_err[d]:.1f}\\%)" for d in pd_order[::-1][:2]), "bestlist": " and ".join(str(d) for d in pd_order[::-1][:2]),
 "gapmax": sci(gap_max) if gap_max > 0 else "0", "gaprel": sci(gap_rel), "recerr": sci(rec),
 "agree": rf(agree, 2), "tprim": rf(tp_tot, 1), "tdual": rf(td_tot, 1),
 "status": "all solver statuses are optimal" if status else "NOT all solver statuses are optimal (check)",
 "trcomment": " (the classes are separable in feature space)" if kn_tr["error"] == 0 else "",
 "nsv": rf(frac_any, 0), "nzero": rf(frac_zero, 0), "shareten": rf(share10, 0),
 "topmis": ("all are classified correctly but lie inside the margin" if top_mis == 0
            else f"{top_mis} of the 16 are misclassified"),
 "solver": "CLARABEL" ,
 "solverdesc": f"CVXPY (solver {"CLARABEL"})",
 "timecomment": ("The dual was " + ("slower" if td_tot > tp_tot else "faster") + " than the primal here "
                 "(both include CVXPY's problem set-up)."),
}
with open("results.tex", "w") as f:
    for k, v in mac.items():
        f.write("\\newcommand{\\r%s}{%s}\n" % (k, v))
with open("keynumbers.csv", "w") as f:
    f.write("key,value\n")
    for k, v in [("student_number", STUDENT),
                 ("model", f"OvR squared-hinge SVM; RBF kernel with gamma={GAMMA:g} and C={CBEST:g}; chosen on {n_vala} validation images; solved via primal and dual"),
                 ("loss", "other"), ("multiclass", "ovr"), ("kernel", "rbf"), ("n_train", NF),
                 ("train_error", rf(kn_tr["error"])), ("val_error", rf(val_err)), ("n_test", len(yt)),
                 
                 ("test_error", rf(te_err)), ("confused_pair", kn_te["confused_pair"]),
                 ("duality_gap", f"{gap_max:.1e}"), ("time_primal", rf(tp_tot, 1)), ("time_dual", rf(td_tot, 1))]:
        f.write(f"{k},{v}\n")
json.dump({k: str(v) for k, v in mac.items()}, open("results.json", "w"), indent=1)
print("\nDone. Wrote results.tex, keynumbers.csv, fig_main.pdf, fig_grid.pdf, fig_hard.pdf")
print("test confusion matrix:\n", M)


Done. Wrote results.tex, keynumbers.csv, fig_main.pdf, fig_grid.pdf, fig_hard.pdf
test confusion matrix:
 [[ 967    0    2    0    0    2    6    1    2    0]
 [   0 1124    2    1    0    2    5    0    1    0]
 [  15    0  974    8    3    0    2   13   16    1]
 [   0    0    9  955    1   11    3    8   19    4]
 [   1    1    3    0  949    0    7    2    3   16]
 [   5    0    3   17    1  846    8    2    4    6]
 [  11    3    1    0    2    6  934    0    1    0]
 [   0   15   18    0    6    0    0  979    1    9]
 [   5    1    5    7    7    8    4   11  924    2]
 [   9    4    1    9   14    6    2    8    9  947]]
